In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder

In [ ]:
train = 'images/train'
test = 'images/test'

In [ ]:
os.chdir(os.path.join(os.getcwd(), '..'))

In [ ]:
def dataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir, label)):
            image_paths.append(os.path.join(dir, label, imagename))
            labels.append(label)
        print(label, 'done')
    return image_paths, labels

traindf = pd.DataFrame()
traindf['image'], traindf['label'] = dataframe(train)
print(traindf)

testdf = pd.DataFrame()
testdf['image'], testdf['label'] = dataframe(train)
print(testdf)

In [ ]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode='grayscale')
        img = np.array(img)
        features.append(img)
    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)
    return features

train_features = extract_features(traindf['image'])
test_features = extract_features(testdf['image'])

In [ ]:
x_train = train_features / 255.0
x_test = test_features / 255.0

In [ ]:
le = LabelEncoder()
le.fit(traindf['label'])

In [ ]:
y_train = le.transform(traindf['label'])
y_test = le.transform(testdf['label'])

y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [ ]:
model = Sequential()

# Convolutional Layers
model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', input_shape=(48, 48, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

# Fully Connected Layers
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# Output Layer
model.add(Dense(7, activation='softmax'))

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(x=x_train, y=y_train, batch_size=128, epochs=30, validation_data=(x_test, y_test))

In [ ]:
model_json = model.to_json()
with open('model/emotiondetector.json', 'w') as json_file:
    json_file.write(model_json)
model.save('model/emotiondetector.h5')

print('Model saved successfully.')

In [ ]:
with open('model/emotiondetector.json', 'r') as json_file:
    model_json = json_file.read()

model = model_from_json(model_json)
model.load_weights('model/emotiondetector.h5')

print('Model loaded successfully.')

In [ ]:
labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

def preprocess_image(image_path):
    img = load_img(image_path, color_mode='grayscale', target_size=(48, 48))
    feature = img_to_array(img)
    feature = feature.reshape(1, 48, 48, 1)
    return feature / 255.0

In [ ]:
image_path = os.path.join("images", "test", "neutral", "PrivateTest_1129340.jpg")
print("Original image is of 'neutral'")

img = preprocess_image(image_path)
pred = model.predict(img)
pred_label = labels[np.argmax(pred)]
print("Model prediction is:", pred_label)

In [ ]:
image_path = os.path.join("images", "train", "sad", "Training_356283.jpg")
print("Original image is of 'sad'")

img = preprocess_image(image_path)
pred = model.predict(img)
pred_label = labels[np.argmax(pred)]
print("Model prediction is:", pred_label)

plt.imshow(img.reshape(48, 48), cmap="gray")
plt.title(f"Prediction: {pred_label}")
plt.axis("off")
plt.show()

In [ ]:
haar_file = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(haar_file)

labels = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "neutral",
    5: "sad",
    6: "surprise"
}

def preprocess_image(image):
    image = cv2.resize(image, (48, 48))
    image = image.astype("float32") / 255.0
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    return image

webcam = cv2.VideoCapture(0)

while True:
    ret, frame = webcam.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        preprocessed_face = preprocess_image(face)

        prediction = model.predict(preprocessed_face)
        emotion_label = labels[np.argmax(prediction)]

        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, emotion_label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

webcam.release()
cv2.destroyAllWindows()